In [1]:
import time
import os
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
# Set up Chrome WebDriver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# Open the BSSDB website
driver.get("https://www.bssdb.dev")

# Maximize the window
driver.maximize_window()

# Wait for the page to load
driver.implicitly_wait(5)

In [3]:
# Create a folder to store images
if not os.path.exists("card_thumbnails"):
    os.makedirs("card_thumbnails")

In [4]:
# Function to download the images from the current page
def download_images():
    # Find the image elements based on the `img` tag with `src` containing the thumb URL
    images = driver.find_elements(By.CSS_SELECTOR, "img[src*='thumb']")  # This will match all images whose 'src' contains 'thumb'

    # Download each image and save to the corresponding folder
    for idx, img in enumerate(images):
        # Get the image URL
        img_url = img.get_attribute("src")
        
        # Extract the part of the URL that is used as the image name (e.g., BSS06-001)
        img_name = img_url.split("/")[-1].split(".")[0]  # Split at "/" to get the last part and remove the ".png"
        
        # Get the folder name from the first part of the image name (before the dash)
        folder_name = img_name.split("-")[0]
        
        # Create the folder if it does not exist
        folder_path = f"card_thumbnails/{folder_name}"
        os.makedirs(folder_path, exist_ok=True)
        
        # Download the image
        img_data = requests.get(img_url).content
        
        # Save the image to the folder using the extracted name
        img_path = os.path.join(folder_path, f"{img_name}.png")
        with open(img_path, "wb") as img_file:
            img_file.write(img_data)
        
        print(f"Downloaded {img_path}")


In [5]:
# Start scraping images from the first page
#download_images()

In [ ]:
# Function to cycle through each card on the page
def cycle_through_cards_on_page():
    try:
        # Wait for the card images to be visible on the page
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")))
        
        # Find all the card images on the page
        card_images = driver.find_elements(By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")
        
        # Loop through each card
        for image in card_images:
            try:
                # Scroll to the card to make sure it's in view
                driver.execute_script("arguments[0].scrollIntoView();", image)

                # Wait until the image is clickable and click it to open the card
                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(image))
                image.click()  # Click to open the card
                time.sleep(2)  # Wait for the card to open

                # Find the close button (anchor element) and click it to close the card
                close_button = driver.find_element(By.CSS_SELECTOR, "button.bp4-button.bp4-small.bp4-intent-danger")
                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(close_button))  # Wait for the close button to be clickable
                close_button.click()  # Click to close the card
                time.sleep(1)  # Allow time for the card to close

            except Exception as e:
                print(f"Error while interacting with a card image: {e}")
        
    except Exception as e:
        print(f"Error while cycling through cards: {e}")


In [7]:
def navigate_pages():
    # First, cycle through cards on the current page
    cycle_through_cards_on_page()

    while True:
        try:
            # Find the "next page" button by targeting the button class and icon
            next_button = driver.find_element(By.CSS_SELECTOR, "button.bp4-button.bp4-outlined.bp4-intent-primary span.bp4-icon-caret-right")
            
            # Scroll the "next" button into view to ensure it's clickable
            driver.execute_script("arguments[0].scrollIntoView();", next_button)

            # Check if the button is disabled (indicating the last page)
            if "bp4-disabled" in next_button.get_attribute("class"):
                print("Reached the last page.")
                break  # Exit the loop when the button is disabled
            
            # Wait until the next button is clickable and click it
            WebDriverWait(driver, 10).until(EC.element_to_be_clickable(next_button))
            next_button.click()
            
            # Wait for the page to load by checking for the presence of image elements
            WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")))

            # After the page loads, cycle through the cards again
            cycle_through_cards_on_page()

            #download_images()

            # Add a slight delay before the next page to avoid overwhelming the server
            time.sleep(2)
        
        except Exception as e:
            # If an error occurs (i.e., the next button is not found or any other issue), break the loop
            print("No more pages or error:", e)
            break


In [8]:
navigate_pages()

Found 18 cards on this page.
Found 18 cards on this page.


KeyboardInterrupt: 

In [9]:
# Close the browser
driver.quit()